# Statistical validation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yazanjer/An_Explainable_AI_Education/blob/main/notebooks/05_statistical_validation.ipynb)

**Answers:** Editor comments 4 and 6
**Estimated runtime:** 2 h quick / 20 h full · **Hardware:** CPU
**Quick mode:** set `QUICK_MODE = True` in the setup cell for a fast smoke test.

Confidence intervals, permutation nulls, feature-selection stability and effect sizes,
computed from the stored per-fold predictions. Nothing is refit here.

**Two permutation nulls, deliberately distinct.** Unrestricted permutation is the
leakage test and must give AUC ~ 0.50. Within-school permutation preserves each
school's class composition, so its expected value is above chance and is *estimated*.
Conflating them produces a false leakage alarm.

---

## Aggregation contract (audit M2)

The previous version of this notebook globbed every `*_preds.parquet` in the
checkpoint directory, concatenated them and grouped by `task` alone — with no
configuration filter, mixing selectors, repeats and plausible values into a single
frame. At full budget that enters each student up to 250 times and produces a
confidence interval with no defensible meaning. (On the current local checkpoint
directory the prediction files happen to share one fingerprint, so the old code
returns a plausible-looking 0.8735; that is luck, not correctness, and it ends the
moment a resumed run writes a second fingerprint — as has already happened to the
fold-result checkpoints.)

`vlpso_xai.evaluation.aggregate` replaces it with an explicit three-level
hierarchy. Only the first level is a pooling operation.

| Level | Unit | Operation | Why |
|---|---|---|---|
| 1 | outer folds within (task, method, pv, repeat) | **concatenate** | The folds partition the sample, so each student appears exactly once. AUC and the school-clustered BCa bootstrap variance are computed here and nowhere else. |
| 2 | repeats within (task, method, pv) | **average the AUCs** | A repeat is a re-partition of the same students, not a new sample. Its spread is partition noise and is reported as a range, never added to the sampling variance. |
| 3 | plausible values within (task, method) | **Rubin's rules** | `U` = mean within-PV sampling variance, `B` = between-PV variance. This is the only step that yields a publishable CI, and the only one that carries the PV measurement error — FMI ≈ 0.56, so it dominates. |
| — | methods | **never combined** | Compared with `contrast_table`, not pooled. |

Selecting the configuration fingerprint is mandatory. If the checkpoint directory
holds more than one, `load_fold_predictions` raises and lists them rather than
guessing.

---


In [ ]:
# --- Environment setup -------------------------------------------------
# Detects Colab, mounts Drive only when in Colab, installs pinned deps.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
QUICK_MODE = True   # set False for the full budget

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT = Path("/content/drive/MyDrive/An_Explainable_AI_Education")
    PROJECT.mkdir(parents=True, exist_ok=True)
    if not (PROJECT / "src").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/yazanjer/An_Explainable_AI_Education.git", str(PROJECT)],
                       check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT / "requirements.txt")], check=False)
else:
    PROJECT = Path(os.environ.get("VLPSO_PROJECT_ROOT", Path.cwd().parent))

os.environ["VLPSO_PROJECT_ROOT"] = str(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))

from vlpso_xai.config import load_config, set_global_seeds, environment_report
cfg = load_config("quick" if QUICK_MODE else "default")
set_global_seeds(cfg.seed)
cfg.paths.mkdirs()
print("project root:", cfg.paths.root)
print("config:", cfg.config_path.name, "| hash:", cfg.hash()[:12])


## 1. Inventory the checkpoints

Print what is actually on disk **before** computing anything. Five of the seven
bugs found in this rebuild were caught by reading printed intermediate
quantities, not by unit tests.


In [ ]:
import pandas as pd
from vlpso_xai.evaluation.aggregate import (
    load_fold_predictions, aggregation_table, aggregate_task_method,
)

CFG_FINGERPRINT = None   # set to a 10-char hash to pin one run; None = require uniqueness

try:
    cset = load_fold_predictions(cfg.paths.checkpoints, cfg_fingerprint=CFG_FINGERPRINT)
except ValueError as exc:
    print(exc)
    raise

print("fingerprint:", cset.fingerprint)
print("tasks      :", cset.tasks)
print("methods    :", cset.methods)
inv = cset.describe()
display(inv)

EXPECTED_FOLDS = int(inv["n_folds"].mode().iloc[0])
ragged = inv[inv["n_folds"] != EXPECTED_FOLDS]
if not ragged.empty:
    print(f"\nWARNING: {len(ragged)} (task, method, pv, rep) cells do not have "
          f"{EXPECTED_FOLDS} folds. These will raise in aggregation rather than "
          f"be scored on a partial sample:")
    display(ragged)

## 2. Level 1-3 aggregation with Rubin's rules

`summary` is the reportable frame: one row per (task, method), with the CI that
carries plausible-value uncertainty. `per_pv` and `per_repeat` are written
alongside it so every published figure has an artefact behind it.

`single_pv = True` marks a quick-mode row where between-imputation variance is
undefined. Those rows are diagnostics and must not reach the manuscript.


In [ ]:
N_RESAMPLES = cfg.section("statistics", "bootstrap", "n_resamples")
BOOT_METHOD = cfg.section("statistics", "bootstrap", "method")
ALPHA       = cfg.section("statistics", "bootstrap", "alpha")

agg = aggregation_table(
    cset,
    metric="auc",
    n_resamples=N_RESAMPLES,
    bootstrap_method=BOOT_METHOD,
    alpha=ALPHA,
    expected_folds=EXPECTED_FOLDS,
)

summary = agg["summary"].sort_values(["task", "method"]).reset_index(drop=True)
display(summary[[
    "task", "method", "n_pv", "n_repeats", "n_folds_per_repeat", "n_students",
    "n_schools", "estimate", "ci_low", "ci_high", "standard_error",
    "within_variance", "between_variance", "fmi", "single_pv",
]])

if summary["single_pv"].any():
    print("\nNOTE: rows with single_pv=True have no between-imputation variance. "
          "Their intervals understate uncertainty and are quick-mode only.")

outdir = cfg.paths.results / "statistics"
outdir.mkdir(parents=True, exist_ok=True)
for name, frame in agg.items():
    frame.to_csv(outdir / f"aggregate_{name}.csv", index=False)
print("written:", sorted(p.name for p in outdir.glob("aggregate_*.csv")))

### 2a. What the old pooled number would have been

Kept as a demonstration, not a result. The gap between the two intervals is the
size of the error M2 describes: the pooled interval is narrower because it counts
each student once per (method x repeat x PV) cell.


In [ ]:
from vlpso_xai.evaluation.metrics import cluster_bootstrap_ci
import glob

_all = pd.concat(
    [pd.read_parquet(p) for p in glob.glob(str(cfg.paths.checkpoints / "*_preds.parquet"))],
    ignore_index=True,
)
rows = []
for task, sub in _all.groupby("task"):
    naive = cluster_bootstrap_ci(
        sub.y_true.to_numpy(), sub.y_score.to_numpy(), sub.group.to_numpy(),
        metric="auc", n_resamples=min(N_RESAMPLES, 500),
    )
    ok = summary[summary.task == task]
    rows.append({
        "task": task,
        "n_rows_pooled": len(sub),
        "n_rows_correct": int(ok["n_students"].max()) if not ok.empty else None,
        "pooled_width": naive["ci_high"] - naive["ci_low"],
        "correct_width": float((ok["ci_high"] - ok["ci_low"]).max()) if not ok.empty else None,
    })
display(pd.DataFrame(rows))
print("The pooled frame mixes", _all.get("cfg_fingerprint", pd.Series(dtype=object)).nunique(),
      "fingerprint(s) and", _all.method.nunique(), "method(s). It is shown for "
      "contrast only and is not written to results/.")

## 3. Permutation nulls

Two nulls, never interchangeable. Unrestricted must return ~0.50; within-school is
*estimated*, not asserted, because it preserves each school's class composition.


In [ ]:
from vlpso_xai.evaluation.permutation import (
    assert_permutation_null_is_chance, decompose_performance,
)

N_PERM = cfg.section("statistics", "permutation", "n_permutations")
print(f"permutations: {N_PERM} (config value; do not reduce for a reported number)")
print("Unrestricted null is asserted against 0.50 +/- "
      f"{cfg.section('statistics', 'permutation', 'tolerance')}.")
print("Within-school null is ESTIMATED. Asserting it against 0.50 produces a "
      "false leakage alarm; that has happened and is documented.")

## 4. Feature-selection stability

In [ ]:
from vlpso_xai.evaluation.stability import stability_table, selection_frequency

sel_path = cfg.paths.results / "selection" / "fold_results.parquet"
if sel_path.exists():
    sel = pd.read_parquet(sel_path)
    st = stability_table(sel[["task", "method", "rep", "fold", "selected"]],
                         all_features=sorted({f for s in sel.selected for f in s}))
    display(st)
else:
    sel = None
    print(f"{sel_path} not found — run notebook 03 first. "
          "Stability and contrasts below are skipped rather than faked.")

## 5. Paired contrasts (audit M4)

Two changes from the previous version.

**The magnitude label is now guarded.** A paired *d* over 3 folds has a standard
error near 0.6, so its interval covers "negligible" and "large" at once. Bands are
emitted only when there are at least `MIN_FOLDS_FOR_MAGNITUDE = 10` matched folds
*and* the bootstrap CI for *d* lies inside a single band. Otherwise the cell reads
`indeterminate (J=3 < 10)` or `indeterminate (CI spans negligible-large)`, and that
string is what belongs in the manuscript table.

**Dropped contrasts are logged.** A skipped pair silently shrinks the multiplicity
family and invalidates every surviving `p_adjusted`.


In [ ]:
from vlpso_xai.evaluation.effect_size import contrast_table, MIN_FOLDS_FOR_MAGNITUDE
import logging; logging.basicConfig(level=logging.WARNING)

if sel is not None:
    lf = sel.rename(columns={"fold": "outer_fold", "rep": "repeat"})
    ct = contrast_table(
        lf, reference="vlpso", metric="auc",
        n_train=int(lf.n_train.mean()), n_test=int(lf.n_test.mean()),
        correction=cfg.section("statistics", "multiplicity", "method"),
    )
    if ct.empty:
        print("No matched contrasts. Nothing to report.")
    else:
        display(ct[["task", "method_b", "n_folds", "mean_difference", "ci_low", "ci_high",
                    "cohens_d_paired", "d_ci_low", "d_ci_high", "magnitude",
                    "underpowered", "p_value", "p_adjusted"]])
        if ct["underpowered"].all():
            print(f"\nEVERY contrast is underpowered at J < {MIN_FOLDS_FOR_MAGNITUDE}. "
                  "The selector comparison cannot support a claim in either "
                  "direction. Re-run notebook 03 at full budget with BPSO and "
                  "VLPSO matched (audit M11) before writing this section.")
        (cfg.paths.results / "statistics").mkdir(parents=True, exist_ok=True)
        ct.to_csv(cfg.paths.results / "statistics" / "contrasts.csv", index=False)